In [1]:
import argparse
import os
import random
import re
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import classification_report, f1_score
from torch.nn.utils.rnn import pack_padded_sequence, pad_sequence
from torch.utils.data import DataLoader, Dataset

PAD, UNK = 0, 1

In [2]:
DRIVE_IDS = {
    "train": {
        "sents.txt": "1nzak5OkrheRV1ltOGCXkT671bmjODLhP",
        "sentiments.txt": "1ye-gOZIBqXdKOoi_YxvpT6FeRNmViPPv",
        "topics.txt": "14MuDtwMnNOcr4z_8KdpxprjbwaQ7lJ_C",
    },
    "dev": {
        "sents.txt": "1sMJSR3oRfPc3fe1gK-V3W5F24tov_517",
        "sentiments.txt": "1GiY1AOp41dLXIIkgES4422AuDwmbUseL",
        "topics.txt": "1DwLgDEaFWQe8mOd7EpF-xqMEbDLfdT-W",
    },
    "test": {
        "sents.txt": "1aNMOeZZbNwSRkjyCWAGtNCMa3YrshR-n",
        "sentiments.txt": "1vkQS5gI0is4ACU58-AbWusnemw7KZNfO",
        "topics.txt": "1_ArMpDguVsbUGl-xSMkTF_p5KpZrmpSB",
    },
}

In [3]:
def download_from_drive(root="UIT-VSFC"):
    import gdown  # Colab có sẵn; nơi khác: pip install gdown

    for split, files in DRIVE_IDS.items():
        os.makedirs(os.path.join(root, split), exist_ok=True)
        for name, file_id in files.items():
            path = os.path.join(root, split, name)
            if not os.path.exists(path):
                gdown.download(id=file_id, output=path, quiet=True)
    return root

In [4]:
def load_data(task, data_dir=None):
    if data_dir is None:
        try:
            from datasets import load_dataset

            ds = load_dataset("uitnlp/vietnamese_students_feedback")
            out = {}
            for ours, theirs in [("train", "train"), ("val", "validation"), ("test", "test")]:
                out[ours] = (list(ds[theirs]["sentence"]), list(ds[theirs][task]))
            return out
        except Exception as e:
            print(f"load_dataset lỗi ({type(e).__name__}), chuyển sang tải từ Google Drive...")
            data_dir = download_from_drive()

    def read(path):
        with open(path, encoding="utf-8") as f:
            return [line.strip() for line in f]

    out = {}
    for ours, folder in [("train", "train"), ("val", "dev"), ("test", "test")]:
        d = os.path.join(data_dir, folder)
        texts = read(os.path.join(d, "sents.txt"))
        labels = [int(x) for x in read(os.path.join(d, f"{task}s.txt"))]
        out[ours] = (texts, labels)
    return out

In [5]:
def tokenize(text):
    return re.findall(r"\w+|[^\w\s]", text.lower())

In [6]:
def build_vocab(texts, min_freq=2):
    cnt = Counter(tok for t in texts for tok in tokenize(t))
    vocab = {"<pad>": PAD, "<unk>": UNK}
    for tok, c in cnt.items():
        if c >= min_freq:
            vocab[tok] = len(vocab)
    return vocab

In [7]:
class TextDS(Dataset):
    def __init__(self, texts, labels, vocab, max_len=100):
        self.items = []
        for t, y in zip(texts, labels):
            ids = [vocab.get(w, UNK) for w in tokenize(t)][:max_len] or [UNK]
            self.items.append((torch.tensor(ids), y))

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        return self.items[i]

In [8]:
def collate(batch):
    seqs, ys = zip(*batch)
    lengths = torch.tensor([len(s) for s in seqs])
    return pad_sequence(seqs, batch_first=True, padding_value=PAD), lengths, torch.tensor(ys)

In [9]:
class GRUClassifier(nn.Module):
    def __init__(self, vocab_size, num_classes, emb_dim=300, hidden=256, layers=5, dropout=0.3):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD)
        self.gru = nn.GRU(
            emb_dim, hidden, num_layers=layers, batch_first=True,
            dropout=dropout,
        )
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden, num_classes)

    def forward(self, x, lengths):
        e = self.drop(self.emb(x))
        packed = pack_padded_sequence(e, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, h_n = self.gru(packed)
        return self.fc(self.drop(h_n[-1]))

In [10]:
@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    preds, gold = [], []
    for x, l, y in loader:
        logits = model(x.to(device), l)
        preds += logits.argmax(1).cpu().tolist()
        gold += y.tolist()
    return gold, preds

def main():
    task = "sentiment" # "topic"
    data_dir = None
    epochs = 15
    batch_size = 64
    lr = 1e-3
    seed = 42

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    data = load_data(task, data_dir)
    vocab = build_vocab(data["train"][0])
    num_classes = len(set(data["train"][1]))
    print(f"Vocab: {len(vocab)} | Number of classes: {num_classes}, | Device: {device}")

    loaders = {
        s: DataLoader(
            TextDS(*data[s], vocab), batch_size=batch_size,
            shuffle=(s == "train"), collate_fn=collate
        )
        for s in data
    }

    model = GRUClassifier(len(vocab), num_classes).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    best_f1, best_state = 0.0, None
    for epoch in range(1, epochs + 1):
        model.train()
        total = 0.0
        for x, l, y in loaders["train"]:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            loss = criterion(model(x, l), y)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total += loss.item()

        gold, pred = evaluate(model, loaders["val"], device)
        f1 = f1_score(gold, pred, average="macro")
        print(f"Epoch {epoch:02d} | loss {total / len(loaders["train"]):.4f} | val macro-f1 {f1:.4f}")
        if f1 > best_f1:
            best_f1 = f1
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    gold, pred = evaluate(model, loaders["test"], device)
    print("\n--------F1 score on test set--------")
    print(f"Macro-F1   : {f1_score(gold, pred, average="macro"):.4f}")
    print(f"Weighted-F1: {f1_score(gold, pred, average="weighted"):.4f}")
    print(classification_report(gold, pred, digits=4))

In [11]:
main()

README.md:   0%|          | 0.00/4.79k [00:00<?, ?B/s]

vietnamese_students_feedback.py:   0%|          | 0.00/4.97k [00:00<?, ?B/s]

load_dataset lỗi (RuntimeError), chuyển sang tải từ Google Drive...
Vocab: 1535 | Number of classes: 3, | Device: cuda
Epoch 01 | loss 0.4366 | val macro-f1 0.7029
Epoch 02 | loss 0.2926 | val macro-f1 0.7512
Epoch 03 | loss 0.2525 | val macro-f1 0.7461
Epoch 04 | loss 0.2179 | val macro-f1 0.7643
Epoch 05 | loss 0.1949 | val macro-f1 0.7917
Epoch 06 | loss 0.1752 | val macro-f1 0.7082
Epoch 07 | loss 0.1521 | val macro-f1 0.7950
Epoch 08 | loss 0.1319 | val macro-f1 0.7809
Epoch 09 | loss 0.1176 | val macro-f1 0.7814
Epoch 10 | loss 0.1004 | val macro-f1 0.7668
Epoch 11 | loss 0.0881 | val macro-f1 0.7891
Epoch 12 | loss 0.0780 | val macro-f1 0.7768
Epoch 13 | loss 0.0786 | val macro-f1 0.7934
Epoch 14 | loss 0.0681 | val macro-f1 0.8048
Epoch 15 | loss 0.0583 | val macro-f1 0.7809

--------F1 score on test set--------
Macro-F1   : 0.7555
Weighted-F1: 0.9015
              precision    recall  f1-score   support

           0     0.9077    0.9489    0.9278      1409
           1     0.